<a href="https://colab.research.google.com/github/mcg-cyber/Metropolia_UAS_ML_Learning/blob/main/6_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Step 0: Install Dependencies


In [1]:
# [Cell 0] Install Dependencies
!pip install -q transformers datasets peft accelerate huggingface_hub gradio

In [2]:
!free -h

               total        used        free      shared  buff/cache   available
Mem:            12Gi       857Mi       8.1Gi       2.0Mi       3.7Gi        11Gi
Swap:             0B          0B          0B


In [3]:
!nvidia-smi

Thu Apr 23 11:47:03 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

Step 1: Environment Setup and Authentication
Before diving into the code, it is important to note that a Hugging Face token is not strictly required to download public models like Qwen. However, if you do not provide one, the Hugging Face library will generate a warning message. Authenticating is a good practice as it suppresses these warnings and is mandatory if you ever decide to use restricted or private models (like Llama 3).

To set this up, add your Hugging Face Access Token to the Secrets tab in Google Colab (the key icon on the left sidebar) and name it HF_TOKEN. Enable notebook access for this secret.

In [4]:
#[Cell 1] Environment Setup & Authentication
from google.colab import userdata
from huggingface_hub import login
import warnings

# Suppress the specific warning about missing tokens
warnings.filterwarnings("ignore", category=UserWarning, module="huggingface_hub.utils._auth")

try:
    # Attempt to fetch the token from Colab's secure storage
    hf_token = userdata.get('HF_TOKEN')

    # Log into the Hugging Face Hub using the retrieved token
    login(hf_token)
    print("Authentication successful.")
except userdata.SecretNotFoundError:
    # If the token isn't found, smoothly continue without crashing
    print("Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.")

Notice: HF_TOKEN not found in Colab secrets. Proceeding without authentication.


Step 2: Load the Starting Model and Tokenizer
What is a Tokenizer? Neural networks cannot read text. The tokenizer translates text strings into numerical IDs (tokens) that the model can process mathematically, and later converts the model's numerical output back into human-readable text.

You can explore how tokenization works in practice using this interactive demo: https://platform.openai.com/tokenizer

In [5]:
#[Cell 2] Load Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load and configure the Tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load and configure the Neural Network Model
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

model.config.use_cache = False

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Step 2b: Baseline Inference (Before Fine-Tuning)
Before training, ask the model a target question. Because the starting model has never seen MediCore.json, it will likely give a generic or hallucinated answer about MediCore Hospital. This establishes a baseline to prove that fine-tuning changed the model's behavior.

In [6]:
# [Cell 2b] Baseline Inference (Before Fine-Tuning)
messages = [{"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}]

# Format the prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)
# print("Token IDs:\n", inputs["input_ids"][0][:20]) # Shows the first ~20 token IDs (numbers)

# Generate an answer using the untrained starting model
output = model.generate(
    **inputs,
    max_new_tokens=50,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Extract and print only the generated response
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("STARTING MODEL ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


STARTING MODEL ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information


Step 3: Load and Preprocess the Dataset
To get the dataset, you have two options: You can either manually upload your MediCore.json file to the Colab file system (using the folder icon on the left sidebar), or let the code automatically download it for you. This step fetches the data and maps it into the standardized ChatML template.

-nc (no-clobber) ensures that if you manually uploaded your own version of MediCore.json, the command will not overwrite it.
-q (quiet) prevents it from printing a messy download progress bar in the Colab output.


In [7]:
# Automatically download the dataset if it hasn't been uploaded manually
!wget -nc -q https://github.com/ML-Course-2026/session6/raw/refs/heads/main/material/datasets/MediCore.json

In [8]:
#[Cell 3] Dataset Loading and Preprocessing
from datasets import load_dataset

raw_data = load_dataset("json", data_files="MediCore.json")

def preprocess(sample):
    messages = [
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]

    # Automatically applies <|im_start|> and <|im_end|> ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

    tokenized = tokenizer(
        text,
        truncation=True,
        #max_length=256,
        padding=False
    )
    # Explicitly create labels for loss calculation
    #tokenized["labels"] = tokenized["input_ids"].copy()
    return tokenized

data = raw_data.map(
    preprocess,
    remove_columns=raw_data["train"].column_names
)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/486 [00:00<?, ? examples/s]

Step 4: Configure PEFT and LoRA Adapters
What is PEFT? PEFT (Parameter-Efficient Fine-Tuning) is both an umbrella concept and an official Hugging Face library. Instead of updating every single parameter in a massive neural network (which requires supercomputers), PEFT methods freeze the original model and only train a tiny fraction of new parameters. The peft library handles all the complex PyTorch code required to do this automatically.

What is LoRA? Large Language Models possess billions of parameters. Updating all of them simultaneously (Full Fine-Tuning) requires immense computing power and VRAM. Low-Rank Adaptation (LoRA) is a technique that freezes the original model weights and injects small, trainable "adapter" matrices into specific layers (like the attention mechanism's q_proj and v_proj). You achieve ~90% of the quality of full fine-tuning while training only ~1% of the parameters. LoRA (Low-Rank Adaptation) is the most popular specific technique inside the PEFT library.

In [9]:
# [Cell 4] LoRA Configuration
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none"
)

model = get_peft_model(model, lora_config)

Step 5: Configure Training Arguments and Execute Training
Understanding the Step Count: If your output shows a particular number of training steps, that number comes from the size of the training split, the batch settings, gradient accumulation, and the number of epochs.

A useful simplified intuition is: Total optimizer steps ≈ (Training Split Size ÷ Effective Batch Size) × Epochs

In this lab, the exact count depends on all of the following:

10% of the dataset is reserved for validation,
per_device_train_batch_size=1,
gradient_accumulation_steps=2,
num_train_epochs=5.
So the exact step count will vary with your dataset size and with how the Trainer rounds partial batches.



In [10]:
#[Cell 5] Training Setup and Execution
from transformers import DataCollatorForLanguageModeling, TrainingArguments, Trainer

# Let the collator handle padding + labels
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# Split 10% of the data for validation
split = data["train"].train_test_split(test_size=0.1)
train_dataset = split["train"]
eval_dataset = split["test"]

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    learning_rate=2e-4,

    per_device_train_batch_size=3,      # ↓ reduce to avoid OOM
    gradient_accumulation_steps=4,      # keeps effective batch size

    fp16=True,                          # ↓ big memory saver

    logging_steps=5,
    eval_strategy="epoch",
    lr_scheduler_type="cosine",
    remove_unused_columns=False
)

# IMPORTANT: enable memory savings
model.gradient_checkpointing_enable()

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    data_collator=data_collator
)

trainer.train()

trainer.save_model("./my_qwen")
tokenizer.save_pretrained("./my_qwen")

Epoch,Training Loss,Validation Loss
1,0.593097,0.618972
2,0.514723,0.624052
3,0.332047,0.680106
4,0.225736,0.760505
5,0.166790,0.817613


('./my_qwen/tokenizer_config.json',
 './my_qwen/chat_template.jinja',
 './my_qwen/tokenizer.json')

Step 6: Load the Fine-Tuned Model
Once training is complete, the LoRA adapters must be loaded alongside the starting model. This cell simulates what you would do if you restarted your Colab notebook and wanted to load your saved work.

Note

If you have NOT restarted your runtime, skip this step. Your model is already in memory from Step 5.

In [11]:
# [Cell 6] Load Model for Testing
from peft import PeftModel, PeftConfig

path = "./my_qwen"
config = PeftConfig.from_pretrained(path)

# 1. Load the original starting-model checkpoint
base_model = AutoModelForCausalLM.from_pretrained(
    config.base_model_name_or_path,
    device_map="cuda",
    torch_dtype=torch.float16
)

# 2. Attach your tiny, fine-tuned adapter to the starting model
model = PeftModel.from_pretrained(base_model, path)

# 3. Re-enable caching for faster inference speeds
model.config.use_cache = True

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Step 7: Test the Fine-Tuned Model
This step tests the model using the proper ChatML format and greedy decoding to retrieve the exact factual data injected during training.



In [12]:
# [Cell 7] Inference Execution
messages =[ {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"} ]

# Format the text with ChatML tags and generation prompt
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# Convert text to tensor numbers and move to GPU
inputs = tokenizer(text, return_tensors="pt").to(model.device)

# Generate the output
output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Strip out the input prompt so we only see the newly generated answer
generated_ids = output[0][inputs.input_ids.shape[1]:]
print("FINE-TUNED ANSWER:\n", tokenizer.decode(generated_ids, skip_special_tokens=True))

FINE-TUNED ANSWER:
 Dr. Samuel Ranta leads the neurology department at MediCore Hospital.


Core project extension: structured JSON and Gradio
Note

The cell numbering intentionally keeps the original working notebook order. Optional merge-and-save remains as Cell 8 in Appendix A at the end, so the main JSON/Gradio extension continues with Cells 9 to 12 here.

Concept: Ensuring Structured Outputs (Markdown or JSON)
When integrating a language model into a user interface like Gradio, you often need the output to be strictly formatted.

Markdown is ideal if you want Gradio to render rich text (bolding, lists, tables).
JSON is ideal if you want Gradio (or another Python script) to programmatically parse the response into dictionaries and variables.
Language models are pattern matchers. To guarantee they output a specific format, you must combine System Prompts with Dataset Formatting.

Important

For the mini project, JSON output and Gradio are part of the required path, not side material.

Strategy 1: Utilize System Prompts
A System Prompt is a special set of instructions given to the model before the user even speaks. It dictates the model's persona and absolute rules. Qwen 2.5 is heavily optimized to obey system prompts.

To ensure formatted output, you must inject this system rule during both training (Step 3) and inference (Step 7).

Here is how you update your inference code (from Step 7) to enforce JSON output using a system role:

In [13]:
#  [Cell 9] [Modified Inference] Enforcing JSON Output
messages = [
    # 1. Add a system prompt with strict formatting rules
    {"role": "system", "content": "You are a helpful assistant. You must ONLY answer in valid JSON format. Do not include any plain text outside the JSON."},

    # 2. Add the user prompt
    {"role": "user", "content": "Who leads the neurology department at MediCore Hospital?"}
]

# Apply the ChatML template (the tokenizer automatically handles the system role)
text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=100,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

generated_ids = output[0][inputs.input_ids.shape[1]:]
response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

print(response_text)

Dr. Elena Varga leads the neurology department at MediCore Hospital.


In [14]:
# How to update Step 3's preprocess function to include a system prompt:
def preprocess(sample):
    messages = [
        {"role": "system", "content": "You are a helpful assistant. You must ONLY answer in valid JSON format."},
        {"role": "user", "content": sample['prompt']},
        {"role": "assistant", "content": sample['completion']}
    ]
    # ... rest of function unchanged

Model-controlled JSON output (via system prompt)
This version relies entirely on the model following instructions.

A strict system prompt tells the model to output JSON.
If the model was trained well, it will follow the format.
If not, the output may break (invalid JSON, extra text, etc.).
Key idea: You are controlling structure through prompting, not code.
Tradeoff: Simple to implement, but not reliable in production.

In [ ]:
#  [Cell 10]
import gradio as gr

# 1. Define the function that Gradio will call when a user submits a prompt
def generate_response(user_prompt):
    messages = [
        # Improved System Prompt: Give the model an exact JSON structure to follow
        {
            "role": "system",
            "content": 'You are a helpful assistant. You must ONLY answer in valid JSON format using the following structure: {"answer": "your detailed response here"}'
        },
        {"role": "user", "content": user_prompt}
    ]

    # Format the text with ChatML tags
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    # Convert text to tensor numbers and move to GPU
    inputs = tokenizer(text, return_tensors="pt").to(model.device)

    # Generate the output
    output = model.generate(
        **inputs,
        max_new_tokens=150,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

    # Strip out the input prompt
    generated_ids = output[0][inputs.input_ids.shape[1]:]
    response_text = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return response_text

# 2. Build the Gradio Interface
demo = gr.Interface(
    fn=generate_response,                      # The function to run
    inputs=gr.Textbox(
        lines=3,
        placeholder="e.g. Who leads the neurology department at MediCore Hospital?",
        label="Enter your prompt here"
    ),
    outputs=gr.Textbox(label="Model Output"),  # Where the output will show
    title="MediCore Fine-Tuned Qwen Bot",
    description="Ask questions about MediCore hospital. The model is instructed to reply in JSON format."
)

# 3. Launch the app (share=True creates a public link you can open)
demo.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://ed875fab2b202aacaf.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
